# Win Probability Model — XGBoost Classifier

Trains a binary XGBoost classifier to predict P(possession team wins)  
given the current game state. Target: `posteam_won` (1/0 based on final score).  

Calibration is checked via a Brier score and reliability diagram.  
Good models achieve Brier ≈ 0.17–0.20 on held-out data.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.calibration import calibration_curve
from pathlib import Path
import sys

pio.renderers.default = 'notebook_connected'
sys.path.insert(0, str(Path('../../').resolve()))

from python.features.engineering import add_features, time_split
from python.features.selection import top_features
from python.models.wp_model import WPModel, WP_FEATURES

DATA = Path('../../data/outputs/play_features.csv')

In [ ]:
df = pd.read_csv(DATA)
df = add_features(df)

# build win outcome label
# nflfastR includes result column: positive = posteam win
if 'result' in df.columns:
    df['posteam_won'] = (df['result'] > 0).astype(int)
elif 'posteam_score_post' in df.columns:
    df['posteam_won'] = (df['posteam_score_post'] > df['defteam_score_post']).astype(int)
else:
    raise ValueError('need result or score columns to build win label')

feat = [c for c in WP_FEATURES if c in df.columns]
train, val = time_split(df)
X_tr, y_tr = train[feat], train['posteam_won']
X_v,  y_v  = val[feat],   val['posteam_won']
print(f'train: {len(X_tr):,}  val: {len(X_v):,}')

In [ ]:
model = WPModel()
model.feature_cols = feat
model.fit_with_eval(X_tr, y_tr, X_v, y_v)

In [ ]:
tr_m = model.evaluate(X_tr, y_tr)
v_m  = model.evaluate(X_v,  y_v)
print(f'train: {tr_m}')
print(f'val:   {v_m}')

In [ ]:
# calibration curve — key quality check for probability models
proba = model.predict_proba(X_v)
frac_pos, mean_pred = calibration_curve(y_v, proba, n_bins=20)

fig = go.Figure()
fig.add_trace(go.Scatter(x=mean_pred, y=frac_pos, mode='lines+markers',
                         name='model', line=dict(color='#2271b3')))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                         name='perfect', line=dict(dash='dash', color='gray')))
fig.update_layout(title='Win Probability — Calibration Curve (2023 val)',
                  xaxis_title='mean predicted probability',
                  yaxis_title='fraction of positives',
                  template='simple_white')
fig.show()

In [ ]:
# feature importance
fi_df = top_features(model.model, feat, n=15)

fig = go.Figure(go.Bar(
    x=fi_df['importance'], y=fi_df['feature'],
    orientation='h', marker_color='#2271b3'
))
fig.update_layout(title='Win Probability — Feature Importance',
                  yaxis={'categoryorder': 'total ascending'},
                  template='simple_white', height=450)
fig.show()

In [ ]:
# save
path = model.save()
print(f'saved to {path}')